In [1]:
import json
import dash
from dash import dcc, html
import plotly.graph_objects as go
import numpy as np
import os

print(os.getcwd())

# Load CTMC data
with open("ctmc_simulation.json", "r") as f:
    json_obj = json.load(f)

# Access the 'data' list inside the JSON
data = json_obj["data"]

# Extract values
time = [d["t"] for d in data]
state = [d["ctmc_state"] for d in data]
rsa_coherence = [d["rsa_coherence"] for d in data]

# Convert to numpy for interpolation / matrix creation
t = np.array(time)
r = np.array(rsa_coherence)

# Create an artificial RSA heatmap over frequency (0–1 Hz)
freq = np.linspace(0, 1, 100)
rsa_matrix = np.array([r * np.exp(-((f - 0.3) ** 2) / 0.1) for f in freq])

# Initialize Dash app
app = dash.Dash(__name__)

app.layout = html.Div(
    [
        html.H1("CTMC-simulering av RSA-Coherence og tilstandsforløp"),
        html.P("Data kombinert fra slp01a–slp02b, modellert som CTMC med 4 stadier."),

        # RSA heatmap
        dcc.Graph(
            id="rsa-heatmap",
            figure=go.Figure(
                data=go.Heatmap(
                    z=rsa_matrix,
                    x=t,
                    y=freq,
                    colorscale="Viridis",
                    colorbar=dict(title="RSA Coherence"),
                )
            ).update_layout(
                title="RSA Coherence over tid og frekvens",
                xaxis_title="Tid (vinduer)",
                yaxis_title="Frekvens (Hz)",
                height=400,
            ),
        ),

        # CTMC state timeline
        dcc.Graph(
            id="state-timeline",
            figure=go.Figure(
                data=go.Scatter(
                    x=t,
                    y=state,
                    mode="lines+markers",
                    line=dict(shape="hv", width=3),
                    marker=dict(size=6, color=state, colorscale="Turbo"),
                )
            ).update_layout(
                title="CTMC tilstandsforløp (1–Apne, 2–Pre-apne, 3–Post-apne, 4–Normal)",
                xaxis_title="Tid (vinduer)",
                yaxis=dict(
                    tickvals=[1, 2, 3, 4],
                    ticktext=["Apne", "Pre-apne", "Post-apne", "Normal"],
                ),
                height=300,
            ),
        ),
    ],
    style = {
    "width": "100vw",       # full viewport width
    "height": "100vh",      # full viewport height
    "margin": "0",          # remove margins
    "fontFamily": "Arial",
    "padding": "20px"
}
,
)

if __name__ == "__main__":
    app.run(debug=True)

c:\Users\Kjæreng\Desktop


In [2]:
import json
import dash
from dash import dcc, html
from dash.dependencies import Input, Output, State
import plotly.graph_objects as go
import numpy as np

# Load CTMC data
with open("ctmc_simulation.json", "r") as f:
    json_obj = json.load(f)
data = json_obj["data"]

# Extract values
time = [d["t"] for d in data]
state = [d["ctmc_state"] for d in data]
rsa_coherence = [d["rsa_coherence"] for d in data]

# Convert to numpy
t = np.array(time)
r = np.array(rsa_coherence)

# Simulated frequency heatmap for visualization
freq = np.linspace(0, 1, 100)
rsa_matrix = np.array([r * np.exp(-((f - 0.3) ** 2) / 0.1) for f in freq])

# Dash app
app = dash.Dash(__name__)

app.layout = html.Div(id="main-div", children=[
    html.H1("Real-time CTMC Simulation of RSA-Coherence"),
    
    html.Div(id="alert-box", style={"color": "white", "fontWeight": "bold", "fontSize": 24, "textAlign": "center"}),

    # RSA heatmap
    dcc.Graph(id="rsa-heatmap"),
    
    # State timeline
    dcc.Graph(id="state-timeline"),

    # Interval for simulation updates
    dcc.Interval(
        id="interval-component",
        interval=250,  # 0.25 seconds per step
        n_intervals=0
    ),

    # Store to keep track of current step
    dcc.Store(id="current-step", data=0)
], style={"width": "90%", "margin": "auto", "fontFamily": "Arial", "padding": "20px"})

@app.callback(
    Output("state-timeline", "figure"),
    Output("rsa-heatmap", "figure"),
    Output("alert-box", "children"),
    Output("main-div", "style"),
    Output("current-step", "data"),
    Input("interval-component", "n_intervals"),
    State("current-step", "data")
)
def update_simulation(n_intervals, current_step):
    if current_step >= len(t):
        return dash.no_update, dash.no_update, "Simulation complete.", dash.no_update, current_step

    # Get current state and time
    current_time = t[current_step]
    current_state = state[current_step]
    current_rsa = r[current_step]

    # Alert text
    alert_text = ""
    style = {"width": "90%", "margin": "auto", "fontFamily": "Arial", "padding": "20px"}

    # Blink red if in state 1 (Apnea)
    if current_state == 1:
        alert_text = f"⚠️ Apnea detected at t={current_time}!"
        if n_intervals % 2 == 0:
            style["backgroundColor"] = "red"
        else:
            style["backgroundColor"] = "white"

    # Update state timeline
    fig_state = go.Figure()
    fig_state.add_trace(go.Scatter(
        x=t[:current_step + 1],
        y=state[:current_step + 1],
        mode="lines+markers",
        line=dict(shape="hv", width=3),
        marker=dict(size=6, color=state[:current_step + 1], colorscale="Turbo")
    ))
    fig_state.update_layout(
        title="Continuous-Time Markov Chain State Timeline",
        xaxis_title="Time",
        yaxis=dict(
            tickvals=[1, 2, 3, 4],
            ticktext=["Apnea", "Pre-apnea", "Post-apnea", "Normal"]
        ),
        height=300
    )

    # Update RSA heatmap
    rsa_partial = np.array([r[:current_step + 1] * np.exp(-((f - 0.3) ** 2) / 0.1) for f in freq])
    fig_heatmap = go.Figure(go.Heatmap(
        z=rsa_partial,
        x=t[:current_step + 1],
        y=freq,
        colorscale="Viridis",
        colorbar=dict(title="RSA Coherence")
    ))
    fig_heatmap.update_layout(
        title="RSA Coherence Over Time",
        xaxis_title="Time",
        yaxis_title="Frequency (Hz)",
        height=400
    )

    # Increment step
    current_step += 1

    return fig_state, fig_heatmap, alert_text, style, current_step

if __name__ == "__main__":
    app.run(debug=True)

In [2]:
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# Load CTMC data
with open("ctmc_simulation.json", "r") as f:
    json_obj = json.load(f)

data = json_obj["data"]
time = [d["t"] for d in data]
state = [d["ctmc_state"] for d in data]
rsa_coherence = [d["rsa_coherence"] for d in data]

t = np.array(time)
r = np.array(rsa_coherence)
freq = np.linspace(0, 1, 100)

# Number of frames
num_frames = 50

# Speed up by taking fewer data points per frame (sample every N steps)
frame_indices = np.linspace(0, len(t)-1, num_frames, dtype=int)

# Create output folder
output_folder = "frames"
os.makedirs(output_folder, exist_ok=True)

# Generate frames
for i, idx in enumerate(frame_indices):
    t_current = t[:idx+1]
    state_current = state[:idx+1]
    r_current = r[:idx+1]
    rsa_matrix = np.array([r_current * np.exp(-((f - 0.3) ** 2) / 0.1) for f in freq])

    # State timeline
    fig_state = go.Figure()
    fig_state.add_trace(go.Scatter(
        x=t_current,
        y=state_current,
        mode="lines+markers",
        line=dict(shape="hv", width=3),
        marker=dict(size=6, color=state_current, colorscale="Turbo")
    ))
    fig_state.update_layout(
        xaxis_title="Time",
        yaxis=dict(
            tickvals=[1,2,3,4],
            ticktext=["Apne", "Pre-apne", "Post-apne", "Normal"]
        ),
        height=300, width=800
    )

    # RSA heatmap
    fig_heatmap = go.Figure(go.Heatmap(
        z=rsa_matrix,
        x=t_current,
        y=freq,
        colorscale="Viridis",
        colorbar=dict(title="RSA Coherence")
    ))
    fig_heatmap.update_layout(height=400, width=800)

    # Combine plots
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        vertical_spacing=0.1,
                        row_heights=[0.4, 0.6])
    for trace in fig_state.data:
        fig.add_trace(trace, row=1, col=1)
    for trace in fig_heatmap.data:
        fig.add_trace(trace, row=2, col=1)
    fig.update_layout(height=700, width=800)

    # Blinking red background and ALERT for Apnea
    if state[idx] == 1:
        if i % 2 == 0:
            fig.update_layout(plot_bgcolor="red", paper_bgcolor="red")
            fig.add_annotation(
                text="⚠️ ALERT! ⚠️",
                xref="paper", yref="paper",
                x=0.5, y=1.05, showarrow=False,
                font=dict(size=36, color="white"),
                align="center"
            )
        else:
            fig.update_layout(plot_bgcolor="black", paper_bgcolor="black")
    else:
        fig.update_layout(plot_bgcolor="black", paper_bgcolor="black")

    # Save frame
    filename = os.path.join(output_folder, f"frame_{i}.png")
    fig.write_image(filename)

print(f"Saved {num_frames} frames to '{output_folder}/'")

KeyboardInterrupt: 

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import dash
from dash import dcc, html
from dash.dependencies import Input, Output, State

# Load CTMC data
with open("ctmc_simulation.json", "r") as f:
    data = json.load(f)["data"]

time = [d["t"] for d in data]
state = [d["ctmc_state"] for d in data]
rsa_coherence = [d["rsa_coherence"] for d in data]

t = np.array(time)
r = np.array(rsa_coherence)
freq = np.linspace(0, 1, 100)

# ----------- DASH LIVE SIMULATION -----------
FAST_FORWARD_STEPS = 20  # number of steps per interval

app = dash.Dash(__name__)

app.layout = html.Div(id="main-div", children=[
    html.H1("Fast-forward CTMC Simulation", style={"color": "white"}),

    html.Div(id="alert-box", style={
        "color": "white",
        "fontWeight": "bold",
        "fontSize": 48,
        "textAlign": "center",
        "marginBottom": "20px"
    }),

    dcc.Graph(id="rsa-heatmap"),
    dcc.Graph(id="state-timeline"),

    dcc.Interval(id="interval-component", interval=50, n_intervals=0),
    dcc.Store(id="current-step", data=0)
], style={
    "width": "100vw",
    "height": "100vh",
    "margin": "0",
    "padding": "20px",
    "fontFamily": "Arial",
    "display": "flex",
    "flexDirection": "column",
    "alignItems": "center",
    "justifyContent": "flex-start",
    "backgroundColor": "black"
})

@app.callback(
    Output("state-timeline", "figure"),
    Output("rsa-heatmap", "figure"),
    Output("alert-box", "children"),
    Output("main-div", "style"),
    Output("current-step", "data"),
    Input("interval-component", "n_intervals"),
    State("current-step", "data")
)
def update_simulation(n_intervals, current_step):
    next_step = current_step + FAST_FORWARD_STEPS
    if next_step >= len(t):
        next_step = len(t) - 1

    current_state = state[next_step]

    style = {"width": "100vw", "height": "100vh", "margin": "0", "padding": "20px",
             "fontFamily": "Arial", "display": "flex", "flexDirection": "column",
             "alignItems": "center", "justifyContent": "flex-start", "backgroundColor": "black"}

    alert_text = ""
    if current_state == 1:
        if n_intervals % 2 == 0:
            style["backgroundColor"] = "red"
            alert_text = "⚠️ ALERT! ⚠️"

    # State timeline
    fig_state = go.Figure()
    fig_state.add_trace(go.Scatter(
        x=t[:next_step+1], y=state[:next_step+1],
        mode="lines+markers",
        line=dict(shape="hv", width=3),
        marker=dict(size=6, color=state[:next_step+1], colorscale="Turbo")
    ))
    fig_state.update_layout(
        title="CTMC State Timeline",
        xaxis_title="Time",
        yaxis=dict(tickvals=[1,2,3,4], ticktext=["Apne","Pre-apne","Post-apne","Normal"]),
        height=300, plot_bgcolor="black", paper_bgcolor="black", font_color="white"
    )

    # RSA heatmap
    rsa_matrix = np.array([r[:next_step+1] * np.exp(-((f-0.3)**2)/0.1) for f in freq])
    fig_heatmap = go.Figure(go.Heatmap(
        z=rsa_matrix, x=t[:next_step+1], y=freq, colorscale="Viridis",
        colorbar=dict(title="RSA Coherence")
    ))
    fig_heatmap.update_layout(
        title="RSA Coherence Over Time",
        xaxis_title="Time", yaxis_title="Frequency (Hz)",
        height=400, plot_bgcolor="black", paper_bgcolor="black", font_color="white"
    )

    new_step = next_step + 1
    return fig_state, fig_heatmap, alert_text, style, new_step

# ----------- FRAME EXPORT FOR OVERLEAF -----------
output_folder = "frames"
os.makedirs(output_folder, exist_ok=True)

num_frames = 50
frame_indices = np.linspace(0, len(t)-1, num_frames, dtype=int)

for i, idx in enumerate(frame_indices):
    t_current = t[:idx+1]
    state_current = state[:idx+1]
    r_current = r[:idx+1]
    rsa_matrix = np.array([r_current * np.exp(-((f-0.3)**2)/0.1) for f in freq])

    # State timeline
    fig_state = go.Figure()
    fig_state.add_trace(go.Scatter(
        x=t_current, y=state_current, mode="lines+markers",
        line=dict(shape="hv", width=3),
        marker=dict(size=6, color=state_current, colorscale="Turbo")
    ))
    fig_state.update_layout(
        xaxis_title="Time",
        yaxis=dict(tickvals=[1,2,3,4], ticktext=["Apne","Pre-apne","Post-apne","Normal"]),
        height=300,width=800
    )

    # RSA heatmap
    fig_heatmap = go.Figure(go.Heatmap(
        z=rsa_matrix, x=t_current, y=freq, colorscale="Viridis",
        colorbar=dict(title="RSA Coherence")
    ))
    fig_heatmap.update_layout(height=400,width=800)

    # Combine plots
    fig = make_subplots(rows=2,cols=1,shared_xaxes=True, vertical_spacing=0.1,row_heights=[0.4,0.6])
    for trace in fig_state.data: fig.add_trace(trace,row=1,col=1)
    for trace in fig_heatmap.data: fig.add_trace(trace,row=2,col=1)
    fig.update_layout(height=700,width=800)

    # Blinking red + ALERT for Apnea
    if state[idx] == 1:
        if i % 2 == 0:
            fig.update_layout(plot_bgcolor="red", paper_bgcolor="red")
            fig.add_annotation(text="⚠️ ALERT! ⚠️",
                               xref="paper", yref="paper",
                               x=0.5, y=1.05, showarrow=False,
                               font=dict(size=36,color="white"),
                               align="center")
        else:
            fig.update_layout(plot_bgcolor="black", paper_bgcolor="black")
    else:
        fig.update_layout(plot_bgcolor="black", paper_bgcolor="black")

    # Save frame
    fig.write_image(os.path.join(output_folder,f"frame_{i}.png"))

print(f"Saved {num_frames} frames to '{output_folder}/'")

# ----------- RUN DASH APP -----------
if __name__ == "__main__":
    app.run(debug=True)